In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit # ensures don't mix series
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

df = pd.read_csv("../data/processed/cleaned_data.csv")

df_s21 = pd.read_csv("../data/processed/S21_E1_cleaned_data.csv")

print(df.head())

print(df_s21.head())

   Series          Contestant  score_ep1  ppt_ep1  score_ep2  ppt_ep2  \
0       1       Frank Skinner         19      3.8         28      2.8   
1       1     Josh Widdicombe         13      2.6         29      2.9   
2       1       Roisin Conaty          7      1.4         28      2.8   
3       1  Romesh Ranganathan         19      3.8         33      3.3   
4       1             Tim Key         17      3.4         35      3.5   

   final_pct_points  
0            0.2133  
1            0.2156  
2            0.1560  
3            0.2133  
4            0.2018  
   Series        Contestant  score_ep1  ppt_ep1
0      21      Amy Gledhill         16      3.2
1      21  Armando Iannucci         11      2.2
2      21       Joanna Page         22      4.4
3      21      Joel Dommett         11      2.2
4      21   Kumail Nanjiani          9      1.8


In [31]:
features = ["score_ep1", "ppt_ep1"] # explanatory variables
target = "final_pct_points"         # scalar response
groups = df["Series"]               # contestants within a series aren't independent, they compete with each other.

In [32]:
# Create a Group shuffle split object using library imported above

# Split the data into training/test sets but make sure that all rows from the same group (Taskmaster series) stay together

# set up
gss = GroupShuffleSplit(
    n_splits=1,        # create *one* train/test split
    test_size=0.2,     # 20% of the series will be held out for testing
    random_state=42    # Fixed seed for debugging
)

# do the split

# gss.split() creates:
# 1. an index identifying the training set rows
# 2. an index identifying the test set rows

# to do this pass to the function:
# df[features]: the input features (Episode 1 data - the explanatory variables)
# df[target]: the target variable (final performance - the scalar response)
# groups: the grouping variable (the series)

# need next to extract the split
# perform the split
train_idx, test_idx = next(
    gss.split(df[features], df[target], groups))

# train_idx and test_idx identify the rows for the training and target sets (finally)

## Explanatory variables
# produce dataframes for the training set using the train_idx
# This contains Ep1 explanatory variables for contestants in the **training** series
X_train = df.loc[train_idx, features]

# Not testing as this is done in other notebooks - I know how well this model performs
# produce dataframes for the test set using the test_idx
# This contains Ep1 explanatory variables for contestants in **test** series
# X_test = df.loc[test_idx, features]

## Scalar responses
# Select the training scalar responses
# These are the outcomes for the **training** series
y_train = df.loc[train_idx, target]

# Not testing as this is done in other notebooks - I know how well this model performs
# Select the test scalar responses
# These are the outcomes for the **test** series
# y_test = df.loc[test_idx, target]

In [33]:
# create a new linear regression model 
# this tries to estimate a value of y (the scalar response - series overall performance) for 
# any value of x (the explanatory variables - performance in ep 1) - need to find a line of best fit.
# there are different methods to try to minimise the 'residual values'

model = LinearRegression()

# get the model to learn from our data using the training data only
model.fit(X_train, y_train)

LinearRegression()

In [34]:
# using the SERIES 21 explanatory variables, use the model to estimate the outcome for each contestant (in y_pred)

y_pred = model.predict(df_s21[features])

# Important - normalise results across each series so % share of points won totals 100%

# Combine test set information into a single df
y_pred_normalised = df_s21.copy()  # start with  features of df_s21
# y_pred_normalised["Contestant"] = df.loc[test_idx, "Contestant"]  # add contestant names
y_pred_normalised["Series"] = 21          # add series info
y_pred_normalised["Predicted_Final_Pct_Raw"] = y_pred                # add model predictions
y_pred_normalised["Predicted_Final_Pct_Norm"] = 0

# How does it look before normalising?
y_pred_normalised_check = y_pred_normalised.groupby("Series")["Predicted_Final_Pct_Raw"].sum()

print("Raw results (summarised):")
display(y_pred_normalised_check)

# normalise results
y_pred_normalised["Predicted_Final_Pct_Norm"] = (y_pred_normalised.groupby("Series")["Predicted_Final_Pct_Raw"].transform(lambda x: x / x.sum()))

# check each series % totals 100 (1.0)
y_pred_normalised_check = y_pred_normalised.groupby("Series")["Predicted_Final_Pct_Norm"].sum()

print("Normalised results (summarised):")
display(y_pred_normalised_check)

# drop all other columns
y_pred_norm = y_pred_normalised["Predicted_Final_Pct_Norm"]

print(y_pred_norm)

Raw results (summarised):


Series
21    0.990877
Name: Predicted_Final_Pct_Raw, dtype: float64

Normalised results (summarised):


Series
21    1.0
Name: Predicted_Final_Pct_Norm, dtype: float64

0    0.206002
1    0.192362
2    0.222369
3    0.192362
4    0.186906
Name: Predicted_Final_Pct_Norm, dtype: float64


In [40]:
# Combine test set information into a single df

test_results = df_s21.copy()                           # start with s21 ep 1 data

test_results["Predicted_Final_Pct_Norm"] = y_pred_norm # add actual final % points

print(test_results)


   Series        Contestant  score_ep1  ppt_ep1  Predicted_Final_Pct_Norm
0      21      Amy Gledhill         16      3.2                  0.206002
1      21  Armando Iannucci         11      2.2                  0.192362
2      21       Joanna Page         22      4.4                  0.222369
3      21      Joel Dommett         11      2.2                  0.192362
4      21   Kumail Nanjiani          9      1.8                  0.186906


In [42]:
# add forecast placement

placement_df = test_results.copy()

placement_df["forecast_placement"] = (
    test_results
    .groupby("Series")["Predicted_Final_Pct_Norm"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print(placement_df)



   Series        Contestant  score_ep1  ppt_ep1  Predicted_Final_Pct_Norm  \
0      21      Amy Gledhill         16      3.2                  0.206002   
1      21  Armando Iannucci         11      2.2                  0.192362   
2      21       Joanna Page         22      4.4                  0.222369   
3      21      Joel Dommett         11      2.2                  0.192362   
4      21   Kumail Nanjiani          9      1.8                  0.186906   

   forecast_placement  
0                   2  
1                   3  
2                   1  
3                   3  
4                   5  
